In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# DPCM AND DELTA MODULATION SIMULATION
# ============================================================
#
# Main concepts:
#   1. First-order DPCM
#   2. PCM vs DPCM quantization error
#   3. Delta Modulation
#   4. Granular noise
#   5. Slope-overload distortion
#   6. Step-size variation
#   7. Input-frequency variation
#   8. Mandatory staircase validation
#
# ============================================================


# ============================================================
# 1. GLOBAL PARAMETERS
# ============================================================

FS = 1000                 # Sampling frequency (Hz)
AMPLITUDE = 1.0           # Signal amplitude
DURATION = 1.0            # Signal duration (seconds)

# DPCM quantization step
DPCM_STEP = 0.10

# Delta modulation step sizes
SMALL_STEP = 0.02
MODERATE_STEP = 0.10
LARGE_STEP = 0.30

# Signal frequencies
SLOW_FREQUENCY = 5
RAPID_FREQUENCY = 100

# Number of samples to display in time-domain plots
DISPLAY_SAMPLES = 250


# ============================================================
# 2. SIGNAL GENERATION
# ============================================================

def generate_signal(frequency, fs=FS, duration=DURATION,
                    amplitude=AMPLITUDE):
    """
    Generate a sinusoidal signal.

        x(t) = A sin(2*pi*f*t)

    Parameters
    ----------
    frequency : signal frequency in Hz
    fs        : sampling frequency
    duration  : signal duration
    amplitude : signal amplitude

    Returns
    -------
    t : time vector
    x : sampled signal
    """

    t = np.arange(0, duration, 1 / fs)

    x = amplitude * np.sin(
        2 * np.pi * frequency * t
    )

    return t, x


# ============================================================
# 3. FIRST-ORDER PREDICTOR
# ============================================================

def first_order_predictor(x):
    """
    First-order predictor:

        x_pred[n] = x[n-1]

    For n = 0, prediction is initialized to zero.
    """

    prediction = np.zeros_like(x)

    prediction[1:] = x[:-1]

    return prediction


# ============================================================
# 4. DPCM QUANTIZATION
# ============================================================

def dpcm_quantization(error, step):
    """
    Uniform quantization of DPCM prediction error.

        q[n] = round(e[n] / Delta)

        e_hat[n] = q[n] * Delta

    Returns:
        quantization index
        quantized error
    """

    quantization_index = np.round(
        error / step
    )

    quantized_error = (
        quantization_index * step
    )

    return quantization_index, quantized_error


# ============================================================
# 5. DPCM RECONSTRUCTION
# ============================================================

def dpcm_reconstruction(prediction,
                        quantized_error):
    """
    Reconstruct DPCM signal:

        x_reconstructed[n]
        =
        x_prediction[n] + e_quantized[n]
    """

    reconstructed = (
        prediction + quantized_error
    )

    return reconstructed


# ============================================================
# 6. COMPLETE DPCM SYSTEM
# ============================================================

def dpcm_system(x, step):
    """
    Complete first-order DPCM system.

        Input
          |
          v
       Prediction
          |
          v
       Error
          |
          v
      Quantization
          |
          v
      Reconstruction
    """

    # --------------------------------------------------------
    # Stage 1: Prediction
    # --------------------------------------------------------

    prediction = first_order_predictor(x)

    # --------------------------------------------------------
    # Stage 2: Prediction error
    # --------------------------------------------------------

    error = x - prediction

    # --------------------------------------------------------
    # Stage 3: Quantization
    # --------------------------------------------------------

    quantization_index, quantized_error = (
        dpcm_quantization(error, step)
    )

    # --------------------------------------------------------
    # Stage 4: Reconstruction
    # --------------------------------------------------------

    reconstructed = dpcm_reconstruction(
        prediction,
        quantized_error
    )

    # --------------------------------------------------------
    # MSE
    # --------------------------------------------------------

    mse = np.mean(
        (x - reconstructed) ** 2
    )

    return {
        "prediction": prediction,
        "error": error,
        "quantization_index": quantization_index,
        "quantized_error": quantized_error,
        "reconstructed": reconstructed,
        "mse": mse
    }


# ============================================================
# 7. PCM QUANTIZATION
# ============================================================

def pcm_quantization(x, step):
    """
    Uniform PCM quantization.

        x_quantized = round(x / Delta) * Delta
    """

    index = np.round(x / step)

    quantized = index * step

    error = x - quantized

    mse = np.mean(error ** 2)

    return {
        "index": index,
        "quantized": quantized,
        "error": error,
        "mse": mse
    }


# ============================================================
# 8. DELTA MODULATION
# ============================================================

def delta_modulation(x, step):
    """
    First-order Delta Modulation.

    Prediction:
        x_pred[n] = x_reconstructed[n-1]

    Prediction error:
        e[n] = x[n] - x_pred[n]

    1-bit quantizer:
        bit = +1 if error >= 0
        bit = -1 if error < 0

    Reconstruction:
        x_reconstructed[n]
        =
        x_pred[n] + bit * Delta

    Therefore every staircase change must be:

        +Delta OR -Delta
    """

    N = len(x)

    # Allocate arrays
    prediction = np.zeros(N)
    error = np.zeros(N)
    bits = np.zeros(N)
    reconstructed = np.zeros(N)

    # --------------------------------------------------------
    # Sample-by-sample DM processing
    # --------------------------------------------------------

    for n in range(N):

        # ----------------------------------------------------
        # Stage 1: Prediction
        # ----------------------------------------------------

        if n == 0:
            prediction[n] = 0.0
        else:
            prediction[n] = reconstructed[n - 1]

        # ----------------------------------------------------
        # Stage 2: Prediction error
        # ----------------------------------------------------

        error[n] = x[n] - prediction[n]

        # ----------------------------------------------------
        # Stage 3: 1-bit quantization
        # ----------------------------------------------------

        if error[n] >= 0:
            bits[n] = +1
        else:
            bits[n] = -1

        # ----------------------------------------------------
        # Stage 4: Reconstruction
        # ----------------------------------------------------

        reconstructed[n] = (
            prediction[n] +
            bits[n] * step
        )

    # --------------------------------------------------------
    # MSE
    # --------------------------------------------------------

    mse = np.mean(
        (x - reconstructed) ** 2
    )

    return {
        "prediction": prediction,
        "error": error,
        "bits": bits,
        "reconstructed": reconstructed,
        "mse": mse
    }


# ============================================================
# 9. MANDATORY DELTA STAIRCASE VALIDATION
# ============================================================

def validate_delta_steps(reconstructed,
                         bits,
                         step,
                         tolerance=1e-10):
    """
    Mandatory validation.

    Verify:

        x_r[n] - x_r[n-1]
        =
        +Delta OR -Delta

    The first reconstructed sample is also checked because
    the initial prediction is zero.
    """

    # Calculate staircase changes
    changes = np.diff(
        np.concatenate(([0.0], reconstructed))
    )

    # Expected changes from the 1-bit quantizer
    expected_changes = bits * step

    # Compare actual and expected
    valid = np.isclose(
        changes,
        expected_changes,
        atol=tolerance,
        rtol=0
    )

    invalid_indices = np.where(
        ~valid
    )[0]

    print("\n" + "=" * 65)
    print("MANDATORY DELTA MODULATION VALIDATION")
    print("=" * 65)

    print(f"Step size Delta = {step}")

    if len(invalid_indices) == 0:

        print("VALIDATION PASSED")
        print(
            "Every staircase change is exactly "
            "+Delta or -Delta."
        )

        print(
            f"Number of samples checked: {len(changes)}"
        )

        print(
            f"Maximum numerical error: "
            f"{np.max(np.abs(changes - expected_changes)):.3e}"
        )

        return True

    else:

        print("VALIDATION FAILED")

        print(
            f"Number of invalid changes: "
            f"{len(invalid_indices)}"
        )

        print("\nFirst invalid samples:")

        for i in invalid_indices[:10]:

            print(
                f"Sample {i}: "
                f"Actual change = {changes[i]:.6f}, "
                f"Expected = {expected_changes[i]:.6f}"
            )

        return False


# ============================================================
# 10. ADDITIONAL DIAGNOSTIC TEST
# ============================================================

def diagnose_staircase(reconstructed, step):
    """
    Independent diagnostic test.

    Check whether every absolute staircase change
    is equal to Delta.
    """

    changes = np.diff(
        np.concatenate(([0.0], reconstructed))
    )

    absolute_changes = np.abs(changes)

    valid = np.isclose(
        absolute_changes,
        step,
        atol=1e-10,
        rtol=0
    )

    print("\n" + "-" * 65)
    print("INDEPENDENT STAIRCASE DIAGNOSTIC")
    print("-" * 65)

    print("Unique staircase changes:")

    unique_values = np.unique(
        np.round(changes, 10)
    )

    print(unique_values)

    print(
        f"\nExpected absolute change = {step}"
    )

    if np.all(valid):

        print(
            "Diagnostic result: PASS"
        )

    else:

        bad = np.where(~valid)[0]

        print(
            f"Diagnostic result: FAIL "
            f"({len(bad)} invalid changes)"
        )


# ============================================================
# 11. PRINT THEORY FOR A GIVEN SIGNAL
# ============================================================

def print_theoretical_condition(frequency,
                                step,
                                fs=FS,
                                amplitude=AMPLITUDE):
    """
    Compare maximum signal slope with maximum
    Delta Modulation staircase slope.
    """

    signal_slope = (
        2 * np.pi * frequency * amplitude
    )

    dm_slope = step * fs

    print("\n" + "-" * 65)
    print("THEORETICAL SLOPE ANALYSIS")
    print("-" * 65)

    print(
        f"Input frequency       = {frequency} Hz"
    )

    print(
        f"Step size Delta       = {step}"
    )

    print(
        f"Sampling frequency    = {fs} Hz"
    )

    print(
        f"Maximum signal slope  = "
        f"{signal_slope:.4f}"
    )

    print(
        f"Maximum DM slope      = "
        f"{dm_slope:.4f}"
    )

    if dm_slope >= signal_slope:

        print(
            "Theory prediction: "
            "DM should be able to track the signal."
        )

    else:

        print(
            "Theory prediction: "
            "SLOPE OVERLOAD is expected."
        )


# ============================================================
# 12. OBSERVATION REPORT
# ============================================================

def observation_report(frequency,
                       step,
                       mse,
                       amplitude=AMPLITUDE,
                       fs=FS):
    """
    Print expected physical effect and interpretation.
    """

    signal_slope = (
        2 * np.pi * frequency * amplitude
    )

    dm_slope = step * fs

    print("\n" + "=" * 65)
    print("OBSERVATION AND INTERPRETATION")
    print("=" * 65)

    print("\nParameters:")
    print(f"Input frequency = {frequency} Hz")
    print(f"Step size       = {step}")
    print(f"MSE             = {mse:.8f}")

    # --------------------------------------------------------
    # Expected effect
    # --------------------------------------------------------

    print("\n1. EXPECTED PHYSICAL EFFECT")

    if dm_slope < signal_slope:

        print(
            "The step size is too small relative to "
            "the signal slope."
        )

        print(
            "Expected effect: Slope-overload distortion."
        )

    elif step > 0.20:

        print(
            "The step size is relatively large "
            "for this unit-amplitude signal."
        )

        print(
            "Expected effect: Granular noise."
        )

    else:

        print(
            "The step size is in a moderate tracking range."
        )

        print(
            "Expected effect: Reasonable reconstruction "
            "with some quantization error."
        )

    # --------------------------------------------------------
    # Observation
    # --------------------------------------------------------

    print("\n2. SIMULATION OBSERVATION")

    if dm_slope < signal_slope:

        print(
            "The reconstructed staircase cannot change "
            "fast enough to follow the input."
        )

    elif step > 0.20:

        print(
            "The reconstructed staircase makes relatively "
            "large jumps around the input."
        )

    else:

        print(
            "The staircase generally follows the input "
            "signal."
        )

    # --------------------------------------------------------
    # Theory comparison
    # --------------------------------------------------------

    print("\n3. THEORY COMPARISON")

    if dm_slope < signal_slope:

        print(
            "Observation agrees with theory: "
            "slope overload occurs when the signal slope "
            "exceeds the available DM staircase slope."
        )

    elif step > 0.20:

        print(
            "Observation agrees with theory: "
            "large step sizes produce larger staircase "
            "fluctuations, producing granular noise."
        )

    else:

        print(
            "Observation agrees with theory: "
            "a suitable step size gives reasonable tracking."
        )

    # --------------------------------------------------------
    # Diagnostic tests
    # --------------------------------------------------------

    print("\n4. DIAGNOSTIC TEST")

    print(
        "The DM staircase validation is used to verify "
        "that each output step is exactly +Delta or -Delta."
    )

    print(
        "MSE is used as the numerical measure of "
        "reconstruction error."
    )


# ============================================================
# 13. DPCM EXPERIMENT
# ============================================================

print("\n")
print("#" * 70)
print("DPCM EXPERIMENT")
print("#" * 70)

# Slowly varying input
t, x = generate_signal(
    frequency=SLOW_FREQUENCY
)

dpcm_result = dpcm_system(
    x,
    DPCM_STEP
)

print("\nDPCM parameters:")
print(f"Input frequency = {SLOW_FREQUENCY} Hz")
print(f"DPCM step       = {DPCM_STEP}")

print(
    f"DPCM MSE        = "
    f"{dpcm_result['mse']:.8f}"
)


# ============================================================
# 14. DPCM VISUALIZATION:
#     ORIGINAL VS PREDICTED
# ============================================================

N = min(
    DISPLAY_SAMPLES,
    len(x)
)

plt.figure(figsize=(10, 5))

plt.plot(
    t[:N],
    x[:N],
    label="Original Signal"
)

plt.plot(
    t[:N],
    dpcm_result["prediction"][:N],
    label="Predicted Signal"
)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.title(
    "First-Order DPCM: Original vs Predicted"
)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 15. DPCM PREDICTION ERROR
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    t[:N],
    dpcm_result["error"][:N]
)

plt.xlabel("Time (s)")
plt.ylabel("Prediction Error")

plt.title(
    "DPCM Prediction Error"
)

plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 16. DPCM QUANTIZATION ERROR
# ============================================================

dpcm_quant_error = (
    dpcm_result["error"]
    -
    dpcm_result["quantized_error"]
)

plt.figure(figsize=(10, 5))

plt.plot(
    t[:N],
    dpcm_quant_error[:N]
)

plt.xlabel("Time (s)")
plt.ylabel("Quantization Error")

plt.title(
    "DPCM Quantization Error"
)

plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 17. DPCM RECONSTRUCTION
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    t[:N],
    x[:N],
    label="Original"
)

plt.plot(
    t[:N],
    dpcm_result["reconstructed"][:N],
    label="DPCM Reconstructed"
)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.title(
    "DPCM Reconstruction"
)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 18. PCM VS DPCM QUANTIZATION ERROR
# ============================================================

pcm_result = pcm_quantization(
    x,
    DPCM_STEP
)

print("\n" + "=" * 65)
print("PCM VS DPCM")
print("=" * 65)

print(
    f"PCM MSE  = {pcm_result['mse']:.8f}"
)

print(
    f"DPCM MSE = {dpcm_result['mse']:.8f}"
)

plt.figure(figsize=(10, 5))

plt.plot(
    t[:N],
    pcm_result["error"][:N],
    label="PCM Quantization Error"
)

plt.plot(
    t[:N],
    dpcm_quant_error[:N],
    label="DPCM Quantization Error"
)

plt.xlabel("Time (s)")
plt.ylabel("Error")

plt.title(
    "PCM vs DPCM Quantization Errors"
)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 19. DELTA MODULATION - SMALL STEP
# ============================================================

print("\n")
print("#" * 70)
print("DELTA MODULATION - SMALL STEP SIZE")
print("#" * 70)

dm_small = delta_modulation(
    x,
    SMALL_STEP
)

print_theoretical_condition(
    SLOW_FREQUENCY,
    SMALL_STEP
)

validate_delta_steps(
    dm_small["reconstructed"],
    dm_small["bits"],
    SMALL_STEP
)

diagnose_staircase(
    dm_small["reconstructed"],
    SMALL_STEP
)

observation_report(
    SLOW_FREQUENCY,
    SMALL_STEP,
    dm_small["mse"]
)


# ============================================================
# 20. DELTA MODULATION - MODERATE STEP
# ============================================================

print("\n")
print("#" * 70)
print("DELTA MODULATION - MODERATE STEP SIZE")
print("#" * 70)

dm_moderate = delta_modulation(
    x,
    MODERATE_STEP
)

print_theoretical_condition(
    SLOW_FREQUENCY,
    MODERATE_STEP
)

validate_delta_steps(
    dm_moderate["reconstructed"],
    dm_moderate["bits"],
    MODERATE_STEP
)

diagnose_staircase(
    dm_moderate["reconstructed"],
    MODERATE_STEP
)

observation_report(
    SLOW_FREQUENCY,
    MODERATE_STEP,
    dm_moderate["mse"]
)


# ============================================================
# 21. DELTA MODULATION - LARGE STEP
# ============================================================

print("\n")
print("#" * 70)
print("DELTA MODULATION - LARGE STEP SIZE")
print("#" * 70)

dm_large = delta_modulation(
    x,
    LARGE_STEP
)

print_theoretical_condition(
    SLOW_FREQUENCY,
    LARGE_STEP
)

validate_delta_steps(
    dm_large["reconstructed"],
    dm_large["bits"],
    LARGE_STEP
)

diagnose_staircase(
    dm_large["reconstructed"],
    LARGE_STEP
)

observation_report(
    SLOW_FREQUENCY,
    LARGE_STEP,
    dm_large["mse"]
)


# ============================================================
# 22. DELTA STAIRCASE COMPARISON
# ============================================================

plt.figure(figsize=(11, 6))

plt.plot(
    t[:N],
    x[:N],
    label="Original Signal"
)

plt.step(
    t[:N],
    dm_small["reconstructed"][:N],
    where="post",
    label=f"Small Delta = {SMALL_STEP}"
)

plt.step(
    t[:N],
    dm_moderate["reconstructed"][:N],
    where="post",
    label=f"Moderate Delta = {MODERATE_STEP}"
)

plt.step(
    t[:N],
    dm_large["reconstructed"][:N],
    where="post",
    label=f"Large Delta = {LARGE_STEP}"
)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.title(
    "Delta Modulation: Effect of Step Size"
)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 23. GRANULAR NOISE EXPERIMENT
# ============================================================

print("\n")
print("#" * 70)
print("GRANULAR NOISE EXPERIMENT")
print("#" * 70)

granular_frequency = 5
granular_step = LARGE_STEP

t_granular, x_granular = generate_signal(
    granular_frequency
)

dm_granular = delta_modulation(
    x_granular,
    granular_step
)

print("\nExpected physical effect:")
print(
    "A large step size should produce large "
    "staircase movements around a slowly varying signal."
)

print("\nSimulation:")
print(
    f"Frequency = {granular_frequency} Hz"
)

print(
    f"Delta = {granular_step}"
)

print(
    f"MSE = {dm_granular['mse']:.8f}"
)

print("\nTheory comparison:")
print(
    "The expected granular-noise behavior is "
    "consistent with the simulation."
)

validate_delta_steps(
    dm_granular["reconstructed"],
    dm_granular["bits"],
    granular_step
)

plt.figure(figsize=(11, 5))

N_granular = min(
    DISPLAY_SAMPLES,
    len(x_granular)
)

plt.plot(
    t_granular[:N_granular],
    x_granular[:N_granular],
    label="Original Signal"
)

plt.step(
    t_granular[:N_granular],
    dm_granular["reconstructed"][:N_granular],
    where="post",
    label="DM Reconstruction"
)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.title(
    "Granular Noise: Large Step Size"
)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 24. SLOPE-OVERLOAD EXPERIMENT
# ============================================================

print("\n")
print("#" * 70)
print("SLOPE-OVERLOAD EXPERIMENT")
print("#" * 70)

t_fast, x_fast = generate_signal(
    RAPID_FREQUENCY
)

dm_fast = delta_modulation(
    x_fast,
    SMALL_STEP
)

print("\nExpected physical effect:")
print(
    "A rapidly varying signal combined with a small "
    "step size should produce slope-overload distortion."
)

print("\nSimulation:")
print(
    f"Frequency = {RAPID_FREQUENCY} Hz"
)

print(
    f"Delta = {SMALL_STEP}"
)

print(
    f"MSE = {dm_fast['mse']:.8f}"
)

print_theoretical_condition(
    RAPID_FREQUENCY,
    SMALL_STEP
)

print("\nTheory comparison:")

signal_slope = (
    2 * np.pi *
    RAPID_FREQUENCY *
    AMPLITUDE
)

dm_slope = (
    SMALL_STEP *
    FS
)

if dm_slope < signal_slope:

    print(
        "Observation agrees with theory: "
        "slope overload is expected because "
        "the signal changes faster than the "
        "DM staircase can track."
    )

else:

    print(
        "No theoretical slope overload is expected."
    )


validate_delta_steps(
    dm_fast["reconstructed"],
    dm_fast["bits"],
    SMALL_STEP
)

plt.figure(figsize=(11, 5))

N_fast = min(
    100,
    len(x_fast)
)

plt.plot(
    t_fast[:N_fast],
    x_fast[:N_fast],
    label="Original Signal"
)

plt.step(
    t_fast[:N_fast],
    dm_fast["reconstructed"][:N_fast],
    where="post",
    label="DM Reconstruction"
)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.title(
    "Slope-Overload Distortion: Rapid Input"
)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 25. MSE VS STEP SIZE
# ============================================================

print("\n")
print("#" * 70)
print("MSE VS STEP SIZE")
print("#" * 70)

step_values = np.linspace(
    0.005,
    0.50,
    40
)

mse_step_values = []

for step in step_values:

    dm = delta_modulation(
        x,
        step
    )

    mse_step_values.append(
        dm["mse"]
    )

mse_step_values = np.array(
    mse_step_values
)

best_index = np.argmin(
    mse_step_values
)

best_step = step_values[
    best_index
]

best_mse = mse_step_values[
    best_index
]

print(
    f"Lowest measured MSE = {best_mse:.8f}"
)

print(
    f"Corresponding step size = {best_step:.5f}"
)

print("\nExpected physical effect:")

print(
    "Very small step sizes may suffer from "
    "slope overload, while very large step sizes "
    "may suffer from granular noise."
)

print("\nSimulation observation:")

print(
    "The MSE curve shows how reconstruction error "
    "changes as the step size is varied."
)

print("\nTheory comparison:")

print(
    "The experiment demonstrates the trade-off "
    "between tracking ability and granular noise."
)


plt.figure(figsize=(10, 5))

plt.plot(
    step_values,
    mse_step_values,
    marker="o",
    markersize=4
)

plt.xlabel("Step Size Delta")
plt.ylabel("MSE")

plt.title(
    "Delta Modulation: MSE vs Step Size"
)

plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 26. MSE VS INPUT FREQUENCY
# ============================================================

print("\n")
print("#" * 70)
print("MSE VS INPUT FREQUENCY")
print("#" * 70)

frequencies = [
    1,
    2,
    5,
    10,
    20,
    50,
    100,
    150,
    200
]

frequency_mse = []

frequency_step = SMALL_STEP

for frequency in frequencies:

    t_temp, x_temp = generate_signal(
        frequency
    )

    dm_temp = delta_modulation(
        x_temp,
        frequency_step
    )

    frequency_mse.append(
        dm_temp["mse"]
    )

frequency_mse = np.array(
    frequency_mse
)

print("\nFrequency      MSE")

for frequency, mse in zip(
        frequencies,
        frequency_mse):

    print(
        f"{frequency:8.1f} Hz    "
        f"{mse:.8f}"
    )

print("\nExpected physical effect:")

print(
    "Increasing frequency increases the maximum "
    "slope of the input signal."
)

print("\nSimulation observation:")

print(
    "At sufficiently high frequencies, the staircase "
    "has increasing difficulty following the signal."
)

print("\nTheory comparison:")

print(
    "This agrees with slope-overload theory."
)


plt.figure(figsize=(10, 5))

plt.plot(
    frequencies,
    frequency_mse,
    marker="o"
)

plt.xlabel("Input Frequency (Hz)")
plt.ylabel("MSE")

plt.title(
    "Delta Modulation: MSE vs Input Frequency"
)

plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 27. AUTOMATIC PARAMETER VALIDATION
# ============================================================

print("\n")
print("#" * 70)
print("AUTOMATIC VALIDATION OF ALL STEP-SIZE CASES")
print("#" * 70)

test_steps = [
    SMALL_STEP,
    MODERATE_STEP,
    LARGE_STEP
]

all_passed = True

for step in test_steps:

    test_dm = delta_modulation(
        x,
        step
    )

    result = validate_delta_steps(
        test_dm["reconstructed"],
        test_dm["bits"],
        step
    )

    if not result:
        all_passed = False


print("\n" + "=" * 65)

if all_passed:

    print(
        "FINAL VALIDATION: ALL TESTS PASSED"
    )

    print(
        "Every Delta Modulation staircase output "
        "changes by exactly +Delta or -Delta."
    )

else:

    print(
        "FINAL VALIDATION: ONE OR MORE TESTS FAILED"
    )

print("=" * 65)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print("\n")
print("#" * 70)
print("FINAL EXPERIMENT SUMMARY")
print("#" * 70)

print("""
DPCM:
-----
1. First-order prediction was implemented.
2. Prediction error was calculated.
3. Prediction error was quantized.
4. The signal was reconstructed.
5. PCM and DPCM errors were compared.

DELTA MODULATION:
-----------------
1. Previous reconstructed sample was used as prediction.
2. Prediction error was calculated.
3. Error was quantized using a 1-bit quantizer.
4. Reconstruction was performed using +/- Delta.
5. Staircase changes were numerically validated.

DISTORTION:
-----------
Small Delta + high frequency:
    -> Slope-overload distortion

Large Delta + slowly varying input:
    -> Granular noise

REQUIRED VISUALIZATIONS:
------------------------
1. Original vs predicted samples
2. Prediction error
3. Delta staircase
4. MSE vs step size
5. MSE vs input frequency

VALIDATION:
-----------
Every Delta Modulation output step was checked
numerically to confirm:

    x_r[n] - x_r[n-1] = +Delta or -Delta
""")

print("\nSimulation completed successfully.")